# 02. Feature Analysis

Run the FeatureEngineer on synthetic data and inspect distributions, correlations, and label balance.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.feature_engineer import FeatureEngineer
from src.data.synthetic import SyntheticLOBGenerator

ob, msg = SyntheticLOBGenerator(n_events=30_000, seed=1).generate()
fe = FeatureEngineer(levels=10)
df = fe.fit_transform(ob, msg)
print('rows:', len(df), 'features:', len(fe.get_feature_names()))
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for col in ['bid_ask_imbalance_l1', 'volume_imbalance_weighted', 'kyle_lambda']:
    if col in df.columns:
        df[col].iloc[:2000].plot(ax=ax, alpha=0.7, label=col)
ax.set_title('Microstructure features over 2k events')
ax.legend(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df['label_direction_10'].value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('Label distribution (horizon=10)')
ax.set_xticklabels(['DOWN', 'STATIONARY', 'UP'], rotation=0)
plt.show()

In [ ]:
corr = df[fe.get_feature_names()].corr().abs()
label_corr = df[fe.get_feature_names() + ['label_direction_10']].corr()['label_direction_10'].drop('label_direction_10').sort_values(key=abs, ascending=False)
label_corr.head(15)